In [18]:
import plotly.graph_objects as go
import pandas as pd
df = pd.read_csv('results.csv')

In [19]:
ranking_df = df.groupby('Model')['F1'].mean().reset_index()
ranking_df = ranking_df.sort_values(by='F1', ascending=False).reset_index(drop=True)

# 3. กำหนดชุดสีตามที่คุณระบุ (เรียงตามอันดับโมเดล)
_MODEL_COLORS = [
    "#2563eb", # blue
    "#dc2626", # red
    "#16a34a", # green
    "#d97706", # amber
    "#7c3aed", # violet
    "#0891b2", # cyan
    "#db2777", # pink
    "#65a30d", # lime
]

# แมตช์สีกับโมเดล (ใช้แค่เท่าที่มีจำนวนโมเดล)
model_colors = _MODEL_COLORS[:len(ranking_df)]

# พลิกข้อมูลเพื่อพล็อต (Plotly จะวาดจากล่างขึ้นบนในแนวนอน)
df_plot = ranking_df.iloc[::-1].reset_index(drop=True)
plot_colors = model_colors[::-1]

# 4. สร้างกราฟ
fig = go.Figure(go.Bar(
    x=df_plot['F1'],
    y=df_plot['Model'],
    orientation='h',
    marker=dict(
        color=plot_colors,
        opacity=0.85, # ปรับความอ่อนเพื่อให้ดูนุ่มนวลขึ้น
        line=dict(color='white', width=1)
    ),
    text=df_plot['F1'].apply(lambda x: f'<b>{x:.2f}</b>'),
    textposition='inside',
    insidetextanchor='end',
    textfont=dict(size=14, color='white'),
    width=0.7 # ความหนาของแท่ง
))

# 5. ปรับแต่ง Layout ให้คล้ายธีมในรูปภาพ
fig.update_layout(
    title=dict(
        text="<b>Model Performance Ranking</b><br><span style='font-size:14px; color:#6b7280;'>Mean F1-score comparison across all models</span>",
        font=dict(size=22, color='#111827'),
        x=0.03,
        y=0.92
    ),
    xaxis=dict(
        title="Mean F1-score (%)",
        showgrid=True,
        gridcolor='#f3f4f6', # เส้น Grid จางๆ
        range=[0, 105],
        dtick=10,
        zeroline=False
    ),
    yaxis=dict(
        title="",
        showgrid=False,
        tickfont=dict(size=14, color='#374151')
    ),
    template="plotly_white",
    margin=dict(l=160, r=40, t=100, b=60),
    height=550,
    plot_bgcolor='white',
    paper_bgcolor='white',
    bargap=0.2
)

# แสดงผล
fig.show()

In [24]:
import numpy as np
import plotly.graph_objects as go
import random

# --- [FIX: Repeated Division & Out of Bounds Bug] ---
# ใช้ copy เพื่อป้องกันไม่ให้รันซ้ำแล้วค่าโดนหารไปเรื่อยๆ
plot_df = df.copy()

# ตรวจสอบและหาร 100 เฉพาะเมื่อค่าเป็นหลักหน่วย/สิบ (ป้องกันการรันซ้ำใน Cell เดียวกัน)
if plot_df['Precision'].max() > 1.05:
    plot_df['Precision'] = plot_df['Precision'] / 100
if plot_df['Recall'].max() > 1.05:
    plot_df['Recall'] = plot_df['Recall'] / 100

# 1. กำหนดค่าพื้นฐาน
_MODEL_COLORS = ["#2563eb", "#dc2626", "#16a34a", "#d97706", "#7c3aed", "#0891b2", "#db2777", "#65a30d"]
_TH_SYMBOLS = {
    "P99 Static": "circle",
    "Sliding Mu+αStd": "diamond",
    "Adaptive-z": "square",
    "Entropy-lock": "x"
}
_TH_SHORT = {
    "P99 Static": "P99",
    "Sliding Mu+αStd": "Slid",
    "Adaptive-z": "Adpt",
    "Entropy-lock": "Entr"
}

fig = go.Figure()

# 2. F1-Isolines (เส้นโค้งพื้นหลัง)
x_iso = np.linspace(0.01, 1, 100)
for f1 in [0.2, 0.4, 0.6, 0.8]:
    # Precision = (f1 * Recall) / (2 * Recall - f1)
    y_iso = (f1 * x_iso) / (2 * x_iso - f1)
    mask = (y_iso >= 0) & (y_iso <= 1)
    
    rx = x_iso[mask]
    py = y_iso[mask]
    
    if len(rx) > 0:
        fig.add_trace(go.Scatter(
            x=rx, y=py,
            mode="lines",
            line=dict(color="#cbd5e1", width=1, dash="dot"),
            showlegend=False, hoverinfo="skip"
        ))
        
        # [FIX: Dynamic Indexing] เลือกจุดกึ่งกลางของเส้นเพื่อแปะป้าย F1 ป้องกัน IndexError
        label_idx = len(rx) // 2 
        fig.add_annotation(
            x=rx[label_idx], y=py[label_idx],
            text=f"F1={f1}", showarrow=False,
            font=dict(size=9, color="#94a3b8"),
            bgcolor="white", opacity=0.8
        )

# 3. Best Zone Shading & Annotation
fig.add_shape(
    type="rect", x0=0.75, y0=0.75, x1=1.0, y1=1.0,
    fillcolor="rgba(34,197,94,0.06)",
    line=dict(color="rgba(34,197,94,0.25)", width=1, dash="dot"),
    layer="below"
)
fig.add_annotation(
    x=0.875, y=0.88,
    text="✦ Best Zone",
    showarrow=False,
    font=dict(size=10, color="rgba(22,163,74,0.8)", weight="bold"),
    xanchor="center",
    opacity=0.6
)

# 4. วาดจุด (ไม่มี Text ในกราฟตามที่ต้องการ)
models = plot_df['Model'].unique()
for i, model_name in enumerate(models):
    model_data = plot_df[plot_df['Model'] == model_name]
    color = _MODEL_COLORS[i % len(_MODEL_COLORS)]
    
    # วนลูปตาม Threshold เพื่อให้สัญลักษณ์ตรงตามที่กำหนด
    for th_key in _TH_SYMBOLS.keys():
        row = model_data[model_data['Threshold'] == th_key]
        if row.empty: continue
        
        row = row.iloc[0]
        # Jitter เล็กน้อยเพื่อให้เห็นขอบขาวเวลาทับกันเป๊ะๆ
        display_x = row['Recall'] + random.uniform(-0.002, 0.002)
        display_y = row['Precision'] + random.uniform(-0.002, 0.002)
        
        fig.add_trace(go.Scatter(
            x=[display_x],
            y=[display_y],
            mode="markers", # เอา +text ออกแล้ว
            name=f"{model_name} · {_TH_SHORT.get(th_key)}",
            legendgroup=model_name,
            legendgrouptitle=dict(text=model_name) if th_key == "P99 Static" else None,
            marker=dict(
                size=14, 
                color=color, 
                symbol=_TH_SYMBOLS.get(th_key),
                opacity=0.9,
                line=dict(width=1.5, color="white")
            ),
            hovertemplate=(
                f"<b>{model_name}</b><br>"
                f"TH: {th_key}<br>"
                f"Recall: %{{x:.3f}}<br>"
                f"Precision: %{{y:.3f}}<extra></extra>"
            )
        ))

# 5. Layout Setup
fig.update_layout(
    height=550,
    template="plotly_white",
    title=dict(
        text="<b>PR Plot — Precision × Recall</b>",
        font=dict(size=16, color="#0f172a"),
        x=0.02
    ),
    xaxis=dict(
        title="Recall", range=[-0.02, 1.05],
        gridcolor="#f1f5f9", tickformat=".0%", zeroline=False
    ),
    yaxis=dict(
        title="Precision", range=[-0.02, 1.05],
        gridcolor="#f1f5f9", tickformat=".0%", zeroline=False
    ),
    legend=dict(
        tracegroupgap=8, 
        itemsizing='constant',
        font=dict(size=10),
        bordercolor="#e2e8f0", borderwidth=1,
        x=1.02, y=1
    ),
    margin=dict(l=60, r=200, t=80, b=80)
)

# สัญลักษณ์ Legend อ้างอิงด้านล่าง
symbol_hint = " &nbsp;·&nbsp; ".join([f"<b>{v.upper()}</b> = {k}" for k, v in _TH_SYMBOLS.items()])
fig.add_annotation(
    text=symbol_hint, xref="paper", yref="paper",
    x=0, y=-0.15, showarrow=False,
    font=dict(size=10, color="#64748b"), xanchor="left"
)

fig.show()

/var/folders/9f/xvcgrxhn1kn5vqzpm9whhs840000gr/T/ipykernel_78112/1871931467.py:36: RuntimeWarning: divide by zero encountered in divide
  y_iso = (f1 * x_iso) / (2 * x_iso - f1)


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# 1. เตรียมข้อมูล (สมมติว่าใช้ df ตัวเดิมที่มี Model และ Threshold)
# เรียงลำดับเพื่อให้กลุ่มโมเดลเดียวกันอยู่ด้วยกัน
df_styled = df.sort_values(['Model', 'F1'], ascending=[True, False])

# 2. กำหนดสีพื้นหลังสำหรับ Model (ให้สีอ่อนๆ ตามธีมที่คุณชอบ)
# แมตช์สีหลักกับชื่อโมเดล (ใช้สีจาก _MODEL_COLORS แต่ทำให้จางลงมากเพื่อเป็นพื้นหลัง)
_MODEL_BG = {
    "CNN-LSTM-AE": "rgba(37, 99, 235, 0.1)",   # blue faint
    "GRU-AE": "rgba(220, 38, 38, 0.1)",        # red faint
    "LSTM-AE": "rgba(22, 163, 74, 0.1)",       # green faint
    "LSTM-Attention-AE": "rgba(217, 119, 6, 0.1)", # amber faint
    "Plain-AE": "rgba(124, 58, 237, 0.1)",     # violet faint
    "Plain-LSTM": "rgba(8, 145, 178, 0.1)"     # cyan faint
}

# 3. สร้างฟังก์ชันสร้าง Bar ในช่อง (ใช้ Unicode หรือการไล่สี)
# แต่ใน Plotly Table วิธีที่คลีนที่สุดคือการคำนวณ 'Fill Color' แบบไล่เฉด (Gradient) 
# หรือใช้สีตัวอักษรช่วย

column_colors = []
for col in df_styled.columns:
    if col == 'Model':
        # ใส่สีตามกลุ่มโมเดล
        column_colors.append([_MODEL_BG.get(m, "white") for m in df_styled['Model']])
    elif col in ['Precision', 'Recall', 'F1', 'Accuracy']:
        # คอลัมน์ตัวเลขหลัก: ใช้สีเขียวอ่อนไล่ระดับตามค่า (Heatmap style)
        c_min, c_max = df_styled[col].min(), df_styled[col].max()
        col_c = []
        for val in df_styled[col]:
            norm = (val - c_min) / (c_max - c_min + 1e-9)
            col_c.append(f'rgba(34, 197, 94, {0.05 + (norm * 0.3)})') # Green intensity
        column_colors.append(col_c)
    else:
        # คอลัมน์อื่นๆ ใช้สีขาวปกติ
        column_colors.append(['white'] * len(df_styled))

# 4. สร้างตาราง
fig = go.Figure(data=[go.Table(
    columnwidth = [130, 110] + [70] * (len(df_styled.columns)-2),
    header=dict(
        values=[f"<b>{c}</b>" for c in df_styled.columns],
        fill_color='#f8fafc',
        align='left',
        font=dict(color='#475569', size=12),
        line_color='#e2e8f0',
        height=35
    ),
    cells=dict(
        values=[df_styled[c] for c in df_styled.columns],
        fill_color=column_colors,
        align='left',
        font=dict(color='#1e293b', size=11),
        line_color='#f1f5f9', # เส้นตารางจางๆ
        height=30,
        # ปรับ format ตัวเลขให้ดูสะอาด
        format=[None, None] + [".2f"] * (len(df_styled.columns)-2) 
    )
)])

# 5. ปรับ Layout ให้ดู Minimal เหมือนในภาพ
fig.update_layout(
    title=dict(
        text="<b>Model Benchmarking Summary</b>",
        font=dict(size=18, color='#0f172a'),
        x=0.01
    ),
    margin=dict(l=10, r=10, t=60, b=10),
    height=25 * len(df_styled) + 150 # ความสูง auto ตามจำนวนแถว
)

fig.show()